# Part E

For each transit-accessibility group, income tercile, and children group, calculate annual parking
expenditure as a share of observed gross income:

(a) averaged over all employees in the group, including non-drivers; and 

(b) averaged over employees who still drive in 2025.

The first quantity describes average exposure across the group; the second describes the
burden among those who actually continue paying. Report the two quantities separately. 

Where
feasible, also show the projected burden under your recommended 2026 tariff, making clear which
behavioural assumptions are used.

## Table

**Required output:** Table E1 with group sample sizes, the two 2025 burden measures, and
the corresponding measure under the recommended tariff if it is projected.

In [1]:
# packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy import stats

In [12]:
# Loading data
DAYS_RULE = "cap"
TRANSIT_PATH = "transit_access.csv"
DATA_PATH = "parking_data_clean.csv"
ta = pd.read_csv(TRANSIT_PATH)
raw = pd.read_csv(DATA_PATH)

In [13]:
def prepare(raw, rule):
    """Same merge as Part B, plus the days rule and a car-user flag."""
    x = (raw.drop(columns=["transit_group", "transit_score"], errors="ignore")
            .merge(ta[["home_commune", "site", "transit_score", "transit_group"]],
                   on=["home_commune", "site"], how="left", validate="m:1"))
    bad = x["car_days_per_week"] > 5
    if rule == "cap":
        x.loc[bad, "car_days_per_week"] = 5.0
    elif rule == "drop":
        x = x[~bad]
    x = x.sort_values(["employee_id", "year"]).reset_index(drop=True)
    x["car_user"] = x["car_days_per_week"] > 0
    x["male"] = (x["gender"] == "M").astype(float)
    return x

df = prepare(raw, DAYS_RULE)

In [14]:
df.head()

,employee_id,site,year,parking_fee_chf,main_mode,uses_car,car_days_per_week,home_commune,home_postcode,distance_km,age,gender,n_children,fte,gross_income_chf,income_imputed,transit_score,transit_group,car_user,male
0,E1000,Lausanne,2024,2.0,car,1,2.2,Lausanne,1004,2.9,23,M,0,0.8,59300.0,False,0.942456,High,True,1.0
1,E1000,Lausanne,2025,4.0,car,1,2.0,Lausanne,1004,2.9,23,M,0,0.8,59300.0,False,0.942456,High,True,1.0
2,E1001,Lausanne,2024,2.0,car,1,3.5,Le Mont-Sur-Lausanne,1052,5.6,51,F,1,1.0,124300.0,False,-0.085964,Medium,True,0.0
3,E1001,Lausanne,2025,4.0,car,1,5.0,Le Mont-Sur-Lausanne,1052,5.6,51,F,1,1.0,124300.0,False,-0.085964,Medium,True,0.0
4,E1002,Lausanne,2024,2.0,car,1,2.6,Chavannes-Près-Renens,1022,7.7,39,M,0,1.0,72800.0,False,-0.270905,Low,True,1.0


In [17]:
d25 = df[df["year"] == 2025].set_index("employee_id")

emp = pd.DataFrame({
    "site": d25["site"],
    "car25": d25["uses_car"],
    "days25": d25["car_days_per_week"],
    "income25": d25["gross_income_chf"],
    "n_children": d25["n_children"],
    "transit_group": d25["transit_group"],
    "transit_score": d25["transit_score"],
    "distance25": d25["distance_km"], "age25": d25["age"], "fte25": d25["fte"],
    "male25": d25["male"],
})

emp["transit_group"] = pd.Categorical(emp["transit_group"], categories=["Low", "Medium", "High"], ordered=True)
emp["income_tercile"] = pd.qcut(emp["income25"], 3, labels=["Low", "Middle", "High"])
emp["children_grp"] = emp["n_children"].map({0: "0", 1: "1", 2: "2+"})
emp["All"] = "All car users"

print(emp["income25"].quantile([1/3, 2/3]))             # must match Part B: 77,500 and 103,200

0.333333     77500.0
0.666667    103200.0
Name: income25, dtype: float64


In [ ]:
# compute annual burden TO BE DONE

def c1_panel(gcol, label):
    L = emp[emp["site"] == "Lausanne"]
    out, idx = [], []
    for g, sub in L.groupby(gcol, observed=True):
        a = sub.loc[sub["car24"] == 1, "days24"].dropna()      # car users in 2024
        b = sub.loc[sub["car25"] == 1, "days25"].dropna()      # car users in 2025
        ma, mb = a.mean(), b.mean()
        el = np.log(mb / ma) / LNR
        se = np.sqrt(a.var(ddof=1) / (len(a) * ma**2) + b.var(ddof=1) / (len(b) * mb**2)) / LNR
        out.append({"Mean days 2024": ma, "Mean days 2025": mb, "Change (days)": mb - ma,
                    "Elasticity": el, "SE": se, "N 2024": len(a), "N 2025": len(b)})
        idx.append(str(g))
    o = pd.DataFrame(out)
    o.index = pd.MultiIndex.from_product([[label], idx])
    return o

C1 = pd.concat([c1_panel("All", "Overall"),
                c1_panel("transit_group", "Transit access"),
                c1_panel("income_tercile", "Income tercile"),
                c1_panel("children_grp", "Number of children")])
print(C1.round(3))

## Interpretation

Interpret the results using both responsiveness and burden. Address the following questions:

• Does a weak response indicate low burden, or could it indicate a lack of alternatives?

• Which groups obtain the largest behavioural benefit, and which groups contribute the most
revenue?

• Are there groups for whom a uniform tariff creates a foreseeable equity or feasibility problem?

Do not infer welfare, savings, or public-transport costs that are not observed in the data.